<p style="background-color:mediumseagreen;font-family:newtimeroman;font-size:200%;color:white;text-align:center;border-radius:60px 20px;"><b>Soldier Race Project</b></p>

**Contains:**

- [Importing Libraries](#import-library)
- [Loading The Dataset](#loading-dataset)
- [Understanding The Dataset & EDA](#eda)
- [Data Visualization](#data-viz)
- [Data Preprocessing](#data-prep)
- [Modeling](#modeling)
- [Comparing Models](#compare-models)
- [Final Model & Prediction](#final-model)
- [SMOTE](#smote)
- [Conclusion](#conclusion)

<h1 style="color: mediumseagreen;">Introduction</h1>

The purpose of this project is to predict the ethnic backgrounds (white, black, and Hispanic) of U.S. military personnel based on their body measurements. The study specifically focuses on these three ethnic groups. By analyzing the relationship between ethnicity and body measurements, the project aims to explore how this data can be utilized in classification algorithms.

<h1 id="import-library" style="color: mediumseagreen;">Importing Libraries</h1>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from skimpy import skim
import missingno as msno

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.compose import make_column_transformer
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.metrics import make_scorer, precision_score, recall_score, accuracy_score, f1_score
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score, auc, roc_curve, average_precision_score, precision_recall_curve
from yellowbrick.classifier import ClassPredictionError, ROCAUC

from sklearn.preprocessing import label_binarize
from itertools import cycle

plt.rcParams["figure.figsize"] = (8,6)

import warnings
warnings.filterwarnings("ignore")
warnings.warn("this will not show")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

<h1 id="loading-dataset" style="color: mediumseagreen;">Loading The Dataset</h1>

**About Dataset**

**ANSUR II Databases: Ethnicity Prediction**

**Overview**
- The ANSUR II (Anthropometric Survey of U.S. Army Personnel) dataset originates from a comprehensive anthropometric survey conducted by the U.S. Army in 2012.

**Dataset Description**

- ANSUR II includes detailed anthropometric measurements and ethnicity information for U.S. Army personnel. The dataset comprises over 100 measurements taken from various body parts, including head, hands, feet, legs, arms, chest, and more.

- The dataset features 93 directly measured anthropometric variables and 15 demographic/administrative variables.

- The male dataset contains measurements from a sample of 4,082 individuals.

- The female dataset contains measurements from a sample of 1,986 individuals.

- In addition to anthropometric and demographic data, the ANSUR II database includes 3D body, foot, and head scans of participants. However, due to privacy concerns, these 3D scan data are not publicly available.

**Participants**
- The dataset includes active-duty Army personnel representing diverse ethnic groups and age ranges. Measurements were collected from both male and female participants.

**Measurements and Ethnicity Relationship**
- Different ethnic groups may exhibit distinct body structures due to genetic and environmental factors. These differences are observable in measurements such as leg length, head shape, and hand or foot dimensions.

**Modeling and Prediction**
- Ethnicity prediction is approached as a classification problem using body measurements in the ANSUR II dataset.
- The model learns the relationship between anthropometric measurements and ethnicity, allowing for predictions on new data.

**Importance and Applications of the Project**
- This research is vital for analyzing the anthropometric data of military personnel to inform the design and production of military equipment, uniforms, and systems. Accurate measurements are critical for ensuring the comfort, mobility, and overall safety of military personnel.

- The findings have potential applications in commercial, industrial, and academic domains.

- Additionally, with advancements in technology, such as laser scanners and sensors, ethnicity prediction could be employed in practical scenarios like automated identification systems in airports.

In [ ]:
from charset_normalizer import from_path

result = from_path("/kaggle/input/ansur-ii/ANSUR II MALE Public.csv").best()
print(result.encoding)

In [ ]:
result = from_path("/kaggle/input/ansur-ii/ANSUR II FEMALE Public.csv").best()
print(result.encoding)

In [ ]:
male_df = pd.read_csv("ANSUR II MALE Public.csv", encoding="cp1250")
female_df = pd.read_csv("ANSUR II FEMALE Public.csv", encoding="ascii")

In [ ]:
male_df.head()

In [ ]:
male_df.tail()

In [ ]:
female_df.head()

In [ ]:
female_df.tail()

In [ ]:
male_df.columns == female_df.columns

In [ ]:
female_df.columns.values[0] = "subjectid"

In [ ]:
male_df.columns == female_df.columns

In [ ]:
df = pd.concat([male_df, female_df], axis=0, ignore_index=True)
df.head()

In [ ]:
df.sample(5)

<h1 id="eda" style="color: mediumseagreen;">Understanding Data & EDA</h1>

In [ ]:
# Some columns in the data set contain self-reported values. Since these values ​​may mislead the model, these columns can be dropped from the data set.
# Also, some columns were dropped from the data set because they were not considered necessary for race estimation.
df.drop(columns=["subjectid", "SubjectNumericRace", "Ethnicity", "Heightin", "Weightlbs", "Date", "Installation", "Component", "Branch", "PrimaryMOS"],
        inplace=True
       )

In [ ]:
df.duplicated().sum()

In [ ]:
df = df[(df["DODRace"] == 1) | (df["DODRace"] == 2) | (df["DODRace"] == 3)]

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.describe(include="object").T

In [ ]:
skim(df)

In [ ]:
df.isnull().sum().sum()

In [ ]:
# Numerical features
numeric_columns = df.select_dtypes(include=['number']).columns

# Calculate the number of rows based on the specified number of columns
num_columns = 5
num_rows = (len(numeric_columns) // num_columns) + (1 if len(numeric_columns) % num_columns != 0 else 0)

# Create a figure with specified size
fig, axes = plt.subplots(num_rows, num_columns, figsize=(16, 4 * num_rows))

# Flatten the axes array for easy iteration
axes = axes.flatten()

# Plot each numeric column
for x, col in enumerate(numeric_columns):
    sns.boxplot(data=df[col], color='skyblue', ax=axes[x])
    axes[x].set_title(col)

# Hide any unused axes (if there are any)
for i in range(x + 1, len(axes)):
    axes[i].axis('off')

plt.tight_layout() 
plt.show()

In [ ]:
columns = numeric_columns

# Calculate the number of rows based on the specified number of columns
num_columns = 5
num_rows = (len(numeric_columns) // num_columns) + (1 if len(numeric_columns) % num_columns != 0 else 0)

# Create a figure with specified size
fig, axes = plt.subplots(num_rows, num_columns, figsize=(16, 4 * num_rows))
axes = axes.flatten()

for i, column in enumerate(columns):
    sns.histplot(df[column], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(column)
    
for i in range(len(columns), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = [col for col in df.columns if str(df[col].dtypes) in ["object", "category", "bool"]]
cat_cols

In [ ]:
def cat_summary(dataframe, col_name):

    print(pd.DataFrame({

        col_name: dataframe[col_name].value_counts(),  # Count of unique values

        "Ratio": 100 * dataframe[col_name].value_counts() / len(dataframe)  # Percentage of each unique value

    }))

    print("------------------------------------------------------------------")



for col in cat_cols:

    cat_summary(df, col)

In [ ]:
def group_by_region(race):
    
    usa_states = [
        "California", "New York", "Texas", "Indiana", "Oklahoma", "Illinois", "Florida", "Minnesota", "Michigan", 
        "Mississippi", "Georgia", "Alabama", "South Carolina", "North Carolina", "Ohio", "Louisiana", "Pennsylvania", 
        "Virginia", "Massachusetts", "Wisconsin", "New Jersey", "Arizona","Missouri","Colorado","Maryland","Tennessee","Kentucky","Washington","Kansas"
        ,"South Dakota", "Iowa", "Arkansas", "Connecticut", "Jamaica", "Nebraska", "Hawaii", "Utah", "West Virginia", "Nevada", "Idaho", "Rhode Island",
        "District of Columbia", "North Dakota","Maine", "Delaware", "New Mexico", "New Hampshire", "Vermont", "Montana", "Alaska", "Wyoming", 
        "US Virgin Islands", "United States"
    ]
    latin_america = [
        "Puerto Rico", "Mexico", "Jamaica", "Dominican Republic", "Colombia", "Haiti", "Panama", "Guam", 
        "Guyana", "Brazil", "Peru", "El Salvador", "Ecuador", "Barbados", "Grenada", "Honduras", "Nicaragua", "Cuba", 
        "Romania", "Liberia", "Portugal", "Bulgaria", "Chile", "Argentina", "Belize", "Costa Rica", "Palau", "Cameroon", 
        "Paraguay", "Venezuela", "Fiji", "Trinidad and Tobago", "Antigua and Barbuda", "Ghana", "Togo", "Ivory Coast", 
        "Guatemala", "Senegal"
    ]
    europe = [
        "Germany", "United Kingdom", "Russia", "France", "Italy", "Poland", "Belgium", "Netherlands", "Romania", 
        "Israel", "Spain", "Denmark"
    ]
    asia = [
        "South Korea", "China", "Japan", "Vietnam", "India", "Nepal", "Taiwan", "Iran", "Sri Lanka", "Korea"
    ]
    africa = [
        "South Africa", "Nigeria", "Liberia", "Kenya", "Ethiopia"
    ]
    

    if race in usa_states:
        return "USA and States"
    elif race in latin_america:
        return "Latin America and Caribbean"
    elif race in europe:
        return "Europe"
    elif race in asia:
        return "Asia"
    elif race in africa:
        return "Africa"
    else:
        return "Other"


df['SubjectsBirthRegion'] = df['SubjectsBirthLocation'].apply(group_by_region)

df.head()

In [ ]:
df.SubjectsBirthRegion.value_counts()

In [ ]:
df.drop("SubjectsBirthLocation", axis = 1, inplace = True)

<h1 id="data-viz" style="color: mediumseagreen;">Data Visualization</h1>

<h2 id="dv1" style="color: limegreen;">Gender Distribution</h2>

In [ ]:
df.Gender.value_counts()

In [ ]:
gender_counts = df['Gender'].value_counts()
gender_counts.plot.pie(autopct='%1.1f%%', colors=['skyblue', 'lightcoral'])
plt.title('Gender Distribution')
plt.ylabel('')
plt.show()

<h2 id="dv2" style="color: limegreen;">Writing Preference Distribution</h2>

In [ ]:
df.WritingPreference.value_counts()

In [ ]:
sns.countplot(x='WritingPreference', data=df, palette='viridis')
plt.title('Writing Preference Distribution')
plt.show()

<h2 id="dv3" style="color: limegreen;">Race Distribution</h2>

**DODRace Values**
- 1 = White
- 2 = Black
- 3 = Hispanic

In [ ]:
df.DODRace.value_counts()

In [ ]:
ax = sns.countplot(x='DODRace', data=df, palette='viridis')

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontsize=11, color='black', padding=2)
    
plt.title('Race Distribution')
plt.show()

<h2 id="dv4" style="color: limegreen;">Birth Location Distribution</h2>

In [ ]:
df.SubjectsBirthRegion.value_counts()

In [ ]:
birthlocations = df.SubjectsBirthRegion.value_counts()

sns.set_style("whitegrid")
plt.figure(figsize=(10,6))
ax = sns.barplot(x=birthlocations.values, y=birthlocations.index, palette="viridis")

for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', fontsize=12, color='black', padding=3)
    
ax.set(xlabel='Number', ylabel='Locations')
plt.title('Birth Location Distribution')
plt.show()

<h2 id="dv5" style="color: limegreen;">Height vs Weight with Gender</h2>

In [ ]:
# Converting stature and weight values ​​into cm and kg
df['stature'] = df['stature'] / 10
df['weightkg'] = df['weightkg'] / 10

In [ ]:
sns.scatterplot(x='stature', y='weightkg', data=df, hue='Gender', palette=['skyblue', 'lightcoral'])
plt.title('Height vs Weight with Gender')
plt.show()

<h1 id="data-prep" style="color: mediumseagreen;">Data Preprocessing</h1>

In [ ]:
df['DODRace'] = df['DODRace'].map({1: 0, 2: 1, 3: 2}) # for some models we shoud start our target from 0 ex: XGBoost

In [ ]:
X = df.drop(columns="DODRace")
y = df.DODRace

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2,stratify=y, random_state=42)

In [ ]:
numeric_features = df.drop("DODRace", axis =1).select_dtypes(include ="number").columns
numeric_features

In [ ]:
cat_features =  df.select_dtypes(include ="object").columns
cat_features

In [ ]:
column_transformer = make_column_transformer(
    (StandardScaler(), numeric_features),       # For Scaling 
    (OneHotEncoder(), cat_features)             # For Encoding 
)

In [ ]:
def eval_metric(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_pred = model.predict(X_test)
    print("Test_Set")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print()
    print("Train_Set")
    print(confusion_matrix(y_train, y_train_pred))
    print(classification_report(y_train, y_train_pred))

<h1 id="modeling" style="color: mediumseagreen;">Modeling</h1>

<h2 id="m1" style="color: limegreen;">Logistic Model</h2>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("logistic",LogisticRegression(class_weight='balanced',max_iter=10000,random_state=42))]

logistic_vanilla = Pipeline(steps=operations)

logistic_vanilla.fit(X_train,y_train)

In [ ]:
eval_metric(logistic_vanilla, X_train, y_train, X_test, y_test)

<h3 style="color: lightgreen;">Logistic Model GridSearchCV</h3>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("logistic",LogisticRegression(class_weight='balanced',max_iter=10000,random_state=42))]

logistic_vanilla = Pipeline(steps=operations)
param_grid = {
    'logistic__C': [0.1, 1, 10, 50, 100], 
    'logistic__penalty': ['l1', 'l2'], 
    'logistic__solver': ['liblinear', 'lbfgs'], 
    #'logistic__class_weight': [None, 'balanced'], 
    'logistic__max_iter': [50, 100, 200]
}
  
cv = StratifiedKFold(n_splits = 10)

logistic_grid = GridSearchCV(estimator = logistic_vanilla,
                             param_grid= param_grid,
                             cv = cv,
                             scoring='recall_weighted',
                             n_jobs= -1,
                             return_train_score= True)

logistic_grid.fit(X_train,y_train)

In [ ]:
logistic_grid.best_params_

In [ ]:
logistic_grid.best_estimator_

In [ ]:
logistic_grid.best_score_

In [ ]:
y_pred_test = logistic_grid.predict(X_test)
y_pred_train = logistic_grid.predict(X_train)

In [ ]:
# Accuracy score
logistic_accuracy_test = accuracy_score(y_test, y_pred_test)
logistic_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score 
logistic_f1_test = f1_score(y_test, y_pred_test, average='weighted')
logistic_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
logistic_recall_test = recall_score(y_test, y_pred_test, average='weighted')
logistic_recall_train = recall_score(y_train, y_pred_train, average='weighted')

logistic_accuracy_test, logistic_accuracy_train, logistic_f1_test, logistic_f1_train, logistic_recall_test, logistic_recall_train

In [ ]:
eval_metric(logistic_grid, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves
log_model = logistic_grid.best_estimator_

visualizer = ROCAUC(log_model, classes=[str(i) for i in range(3)])

visualizer.fit(X_train, y_train)          # Fit the training data to the visualizer
visualizer.score(X_test, y_test)          # Evaluate the model on the test data
visualizer.show();                        # Finalize and render the figure

In [ ]:
# Another way to plotting ROC Curves

# Converting the target variable to binary format
y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

# Getting the prediction probabilities
y_pred_proba = logistic_grid.predict_proba(X_test)

# Calculating ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plotting ROC Curves
colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Converting the target variable to binary format
y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

# Getting the prediction probabilities
y_pred_proba = logistic_grid.predict_proba(X_test)

# Calculating Precision recall curve and average precision (AP) for each class
precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])

# Plotting Precision-Recall Curves
colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h2 id="m2" style="color: limegreen;">SVC Model</h2>

In [ ]:
operations = [ ("transformer", column_transformer), 
              ("svc", SVC(random_state=42, probability=True))]

svc_vanilla = Pipeline(steps=operations)

svc_vanilla.fit(X_train,y_train)

In [ ]:
eval_metric(svc_vanilla, X_train, y_train, X_test, y_test)

<h3 style="color: lightgreen;">SVC Model GridSearchCV</h3>

In [ ]:
param_grid = {
    'svc__C': [0.1, 0.5, 1],
    'svc__kernel': ['linear', 'rbf', 'sigmoid'],
    'svc__gamma': ["scale", "auto", 0.1, 1],
    'svc__degree': [1, 2, 3]
}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
svc_grid_model = GridSearchCV(svc_vanilla,
                             param_grid,
                             cv = cv,
                             scoring = "recall_weighted",
                             n_jobs = -1,
                             verbose=2
                             )

svc_grid_model.fit(X_train, y_train)

In [ ]:
svc_grid_model.best_params_

In [ ]:
svc_grid_model.best_estimator_

In [ ]:
svc_grid_model.best_score_

In [ ]:
y_pred_test = svc_grid_model.predict(X_test)
y_pred_train = svc_grid_model.predict(X_train)

In [ ]:
# Accuracy score
svc_accuracy_test = accuracy_score(y_test, y_pred_test)
svc_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
svc_f1_test = f1_score(y_test, y_pred_test, average='weighted')
svc_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
svc_recall_test = recall_score(y_test, y_pred_test, average='weighted')
svc_recall_train = recall_score(y_train, y_pred_train, average='weighted')

svc_accuracy_test, svc_accuracy_train, svc_f1_test, svc_f1_train, svc_recall_test, svc_recall_train

In [ ]:
eval_metric(svc_grid_model, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = svc_grid_model.predict_proba(X_test)

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves
y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]


y_pred_proba = svc_grid_model.predict_proba(X_test)


precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])


colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h2 id="m3" style="color: limegreen;">Random Forest Model</h2>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("RF_model",RandomForestClassifier(class_weight='balanced',random_state=42))]

rf_vanilla = Pipeline(steps= operations).fit(X_train,y_train)

In [ ]:
eval_metric(rf_vanilla, X_train, y_train, X_test, y_test)

<h3 style="color: lightgreen;">RF Model GridSearchCV</h3>

In [ ]:
param_grid = {'RF_model__n_estimators':[150, 200, 250, 300],
             'RF_model__max_features':[2, 4, 'sqrt'],
             'RF_model__max_depth':[3, 4, 5, 6],
             'RF_model__min_samples_split':[1, 2, 3, 4],
             'RF_model__min_samples_leaf': [2, 3, 4],
             'RF_model__max_samples':[0.5, 0.8, 1]}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("RF_model",RandomForestClassifier(class_weight='balanced',random_state=42))]

rf_vanilla = Pipeline(steps= operations)

rf_grid_model = GridSearchCV(rf_vanilla,
                             param_grid,
                             cv = cv,
                             scoring = "recall_weighted",
                             n_jobs = -1, verbose=2)

rf_grid_model.fit(X_train, y_train)

In [ ]:
rf_grid_model.best_params_

In [ ]:
rf_grid_model.best_estimator_

In [ ]:
rf_grid_model.best_score_

In [ ]:
y_pred_test = rf_grid_model.predict(X_test)
y_pred_train = rf_grid_model.predict(X_train)

In [ ]:
# Accuracy score
rf_accuracy_test = accuracy_score(y_test, y_pred_test)
rf_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
rf_f1_test = f1_score(y_test, y_pred_test, average='weighted')
rf_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
rf_recall_test = recall_score(y_test, y_pred_test, average='weighted')
rf_recall_train = recall_score(y_train, y_pred_train, average='weighted')

rf_accuracy_test, rf_accuracy_train, rf_f1_test, rf_f1_train, rf_recall_test, rf_recall_train

In [ ]:
eval_metric(rf_grid_model, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]


y_pred_proba = rf_grid_model.predict_proba(X_test)


fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])


colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]


y_pred_proba = rf_grid_model.predict_proba(X_test)


precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])


colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h2 id="m4" style="color: limegreen;">Boosting Methods</h2>

<h3 style="color: lightgreen;">AdaBoost Model</h3>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("AdaBoost_model",AdaBoostClassifier(random_state=42))]

ada_vanilla = Pipeline(steps= operations)

ada_vanilla.fit(X_train, y_train)

In [ ]:
eval_metric(ada_vanilla, X_train, y_train, X_test, y_test)

<h4 style="color: lightgreen;">AdaBoost Model GridSearchCV</h4>

In [ ]:
param_grid = {
    "AdaBoost_model__n_estimators": [100, 200, 250],
    "AdaBoost_model__learning_rate": [0.01, 0.1, 0.5, 1]
}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
ada_grid_model = GridSearchCV(ada_vanilla, param_grid,
                              cv= cv,
                              scoring= "recall_weighted"
                             )
ada_grid_model.fit(X_train, y_train)

In [ ]:
ada_grid_model.best_params_

In [ ]:
ada_grid_model.best_estimator_

In [ ]:
ada_grid_model.best_score_

In [ ]:
y_pred_test = ada_grid_model.predict(X_test)
y_pred_train = ada_grid_model.predict(X_train)

In [ ]:
# Accuracy score
ada_accuracy_test = accuracy_score(y_test, y_pred_test)
ada_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
ada_f1_test = f1_score(y_test, y_pred_test, average='weighted')
ada_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
ada_recall_test = recall_score(y_test, y_pred_test, average='weighted')
ada_recall_train = recall_score(y_train, y_pred_train, average='weighted')

ada_accuracy_test, ada_accuracy_train, ada_f1_test, ada_f1_train, ada_recall_test, ada_recall_train

In [ ]:
eval_metric(ada_grid_model, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = ada_grid_model.predict_proba(X_test)

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = ada_grid_model.predict_proba(X_test)

precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h3 style="color: lightgreen;">Gradient Boost Model</h3>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("Gradient_model",GradientBoostingClassifier(random_state=42))]

grad_vanilla = Pipeline(steps= operations)

grad_vanilla.fit(X_train, y_train)

In [ ]:
eval_metric(grad_vanilla, X_train, y_train, X_test, y_test)

<h4 style="color: lightgreen;">Gradient Boost Model GridSearchCV</h4>

In [ ]:
param_grid = {
    "Gradient_model__n_estimators": [100, 200, 300],
    "Gradient_model__subsample": [0.5, 0.8],
    "Gradient_model__max_features": [None, 2, 34],
    "Gradient_model__learning_rate": [0.01, 0.1, 0.5],
    'Gradient_model__max_depth': [3, 4, 5]
}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
gb_grid = GridSearchCV(grad_vanilla,
                       param_grid,
                       cv= cv,
                       scoring="recall_weighted",
                       verbose=2,
                       n_jobs=-1,
                       return_train_score=True)

gb_grid.fit(X_train, y_train)

In [ ]:
gb_grid.best_params_

In [ ]:
gb_grid.best_estimator_

In [ ]:
gb_grid.best_score_

In [ ]:
y_pred_test = gb_grid.predict(X_test)
y_pred_train = gb_grid.predict(X_train)

In [ ]:
# Accuracy score
gb_accuracy_test = accuracy_score(y_test, y_pred_test)
gb_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
gb_f1_test = f1_score(y_test, y_pred_test, average='weighted')
gb_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
gb_recall_test = recall_score(y_test, y_pred_test, average='weighted')
gb_recall_train = recall_score(y_train, y_pred_train, average='weighted')

gb_accuracy_test, gb_accuracy_train, gb_f1_test, gb_f1_train, gb_recall_test, gb_recall_train

In [ ]:
eval_metric(gb_grid, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = gb_grid.predict_proba(X_test)

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = gb_grid.predict_proba(X_test)

precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h3 style="color: lightgreen;">XGBoost Model</h3>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("xgb_model", XGBClassifier(random_state = 42))]

xgb_vanilla = Pipeline(steps = operations)

xgb_vanilla.fit(X_train, y_train)

In [ ]:
eval_metric(xgb_vanilla, X_train, y_train, X_test, y_test)

<h4 style="color: lightgreen;">XGB Model GridSearchCV</h4>

In [ ]:
param_grid = {
    "xgb_model__n_estimators": [80, 100, 150],
    'xgb_model__max_depth': [2, 3, 5],
    "xgb_model__learning_rate": [0.05, 0.1],
    "xgb_model__subsample": [0.5, 0.8, 1],
    "xgb_model__colsample_bytree": [0.5, 0.7, 1]
}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
xgb_grid_model = GridSearchCV(xgb_vanilla,
                       param_grid,
                       cv=cv, 
                       scoring="recall_weighted", 
                       n_jobs=-1, 
                       verbose=2, 
                       return_train_score=True)

xgb_grid_model.fit(X_train, y_train)

In [ ]:
xgb_grid_model.best_params_

In [ ]:
xgb_grid_model.best_estimator_

In [ ]:
xgb_grid_model.best_score_

In [ ]:
y_pred_test = xgb_grid_model.predict(X_test)
y_pred_train = xgb_grid_model.predict(X_train)

In [ ]:
# Accuracy score
xgb_accuracy_test = accuracy_score(y_test, y_pred_test)
xgb_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
xgb_f1_test = f1_score(y_test, y_pred_test, average='weighted')
xgb_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
xgb_recall_test = recall_score(y_test, y_pred_test, average='weighted')
xgb_recall_train = recall_score(y_train, y_pred_train, average='weighted')

xgb_accuracy_test, xgb_accuracy_train, xgb_f1_test, xgb_f1_train, xgb_recall_test, xgb_recall_train

In [ ]:
eval_metric(xgb_grid_model, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = xgb_grid_model.predict_proba(X_test)

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = xgb_grid_model.predict_proba(X_test)

precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h3 style="color: lightgreen;">LightGBM Model</h3>

In [ ]:
operations = [ ("transformer",column_transformer), 
              ("LGBM_model", LGBMClassifier(random_state = 42))]

lgbm_vanilla = Pipeline(steps = operations)

lgbm_vanilla.fit(X_train, y_train)

In [ ]:
eval_metric(lgbm_vanilla, X_train, y_train, X_test, y_test)

<h4 style="color: lightgreen;">LightGBM Model GridSearchCV</h4>

In [ ]:
param_grid = {
    'LGBM_model__learning_rate': [0.1, 1],
    'LGBM_model__n_estimators': [50, 100, 200],
    'LGBM_model__num_leaves': [20, 30, 50],  
    'LGBM_model__max_depth': [-1, 3, 4, 5],     
    'LGBM_model__subsample': [0.8, 1.0],
    'LGBM_model__colsample_bytree': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
lgbm_grid_model = GridSearchCV(estimator=lgbm_vanilla,
                         param_grid=param_grid,
                         cv=cv,
                         scoring='recall_weighted',
                         n_jobs=-1,
                         return_train_score=True)

lgbm_grid_model.fit(X_train, y_train)

In [ ]:
lgbm_grid_model.best_params_

In [ ]:
lgbm_grid_model.best_estimator_

In [ ]:
lgbm_grid_model.best_score_

In [ ]:
y_pred_test = lgbm_grid_model.predict(X_test)
y_pred_train = lgbm_grid_model.predict(X_train)

In [ ]:
# Accuracy score
lgbm_accuracy_test = accuracy_score(y_test, y_pred_test)
lgbm_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
lgbm_f1_test = f1_score(y_test, y_pred_test, average='weighted')
lgbm_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
lgbm_recall_test = recall_score(y_test, y_pred_test, average='weighted')
lgbm_recall_train = recall_score(y_train, y_pred_train, average='weighted')

lgbm_accuracy_test, lgbm_accuracy_train, lgbm_f1_test, lgbm_f1_train, lgbm_recall_test, lgbm_recall_train

In [ ]:
eval_metric(lgbm_grid_model, X_train, y_train, X_test, y_test)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = lgbm_grid_model.predict_proba(X_test)

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = lgbm_grid_model.predict_proba(X_test)

precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h3 style="color: lightgreen;">CatBoost Model</h3>

In [ ]:
df2 = df.copy()
df2.head()

In [ ]:
X2 = df2.drop(columns=["DODRace"])
y2 = df2["DODRace"]

In [ ]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, stratify=y2, random_state=42)

In [ ]:
# Scaling numeric columns
scaler = StandardScaler()
X_train_scaled = X_train2.copy()
X_test_scaled = X_test2.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train2[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test2[numeric_features])

In [ ]:
categorical_features = list(X_train_scaled.select_dtypes(include="object").columns)
categorical_features

In [ ]:
train_pool = Pool(X_train_scaled, y_train2, cat_features = categorical_features)
test_pool = Pool(X_test_scaled, y_test2, cat_features = categorical_features)

In [ ]:
cat_vanilla = CatBoostClassifier(random_state=42)

cat_vanilla.fit(X_train_scaled, y_train, cat_features=categorical_features)

In [ ]:
eval_metric(cat_vanilla, X_train_scaled, y_train2, X_test_scaled, y_test2)

<h4 style="color: lightgreen;">CatBoost Model GridSearchCV</h4>

In [ ]:
param_grid = {
    'learning_rate': [0.05, 0.1],
    'iterations': [200, 300],
    'depth': [4, 6],
    'l2_leaf_reg': [3, 5],
    'bagging_temperature': [0.0, 1.0],
    'grow_policy': ['Depthwise'],
    'early_stopping_rounds': [50]
}

cv = StratifiedKFold(n_splits = 10)

In [ ]:
cat_grid_model = GridSearchCV(estimator=cat_vanilla,
                        param_grid=param_grid,
                        cv=cv,
                        scoring='recall_weighted',
                        n_jobs=-1,
                        return_train_score=True
                        )
cat_grid_model.fit(X_train_scaled, y_train2, cat_features= categorical_features)

In [ ]:
cat_grid_model.best_params_

In [ ]:
cat_grid_model.best_score_

In [ ]:
y_pred_test = cat_grid_model.predict(test_pool)
y_pred_train = cat_grid_model.predict(train_pool)

In [ ]:
# Accuracy score
cat_accuracy_test = accuracy_score(y_test, y_pred_test)
cat_accuracy_train = accuracy_score(y_train, y_pred_train)

# F1 score
cat_f1_test = f1_score(y_test, y_pred_test, average='weighted')
cat_f1_train = f1_score(y_train, y_pred_train, average='weighted')

# Recall score
cat_recall_test = recall_score(y_test, y_pred_test, average='weighted')
cat_recall_train = recall_score(y_train, y_pred_train, average='weighted')

cat_accuracy_test, cat_accuracy_train, cat_f1_test, cat_f1_train, cat_recall_test, cat_recall_train

In [ ]:
def eval_metric2(model, X_train, y_train, X_test, y_test, cat_features):
    """
    Prints confusion matrix and classification report for the CatBoost model on the training and test sets.
    
    Args:
        model: Trained CatBoost model.
        X_train: Training dataset.
        y_train: Training labels.
        X_test: Test dataset.
        y_test: Test labels.
        cat_features: Names of categorical features.
    """
    
    # Training predictions
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    y_train_pred = model.predict(train_pool)
    
    # Test predictions
    test_pool = Pool(X_test, y_test, cat_features=cat_features)
    y_pred = model.predict(test_pool)
    
    print("Test_Set")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print()
    print("Train_Set")
    print(confusion_matrix(y_train, y_train_pred))
    print(classification_report(y_train, y_train_pred))

In [ ]:
eval_metric2(cat_grid_model, X_train_scaled, y_train2, X_test_scaled, y_test2, categorical_features)

In [ ]:
# Plotting ROC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = cat_grid_model.predict_proba(X_test_scaled)

fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(i, roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Plotting PRC Curves

y_test_binarized = label_binarize(y_test, classes=np.unique(y_test))
n_classes = y_test_binarized.shape[1]

y_pred_proba = cat_grid_model.predict_proba(X_test_scaled)

precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_binarized[:, i], y_pred_proba[:, i])
    average_precision[i] = average_precision_score(y_test_binarized[:, i], y_pred_proba[:, i])

colors = cycle(['blue', 'red', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(recall[i], precision[i], color=color, lw=2,
             label='PR curve of class {0} (AP = {1:0.2f})'.format(i, average_precision[i]))

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

<h2 id="m5" style="color: limegreen;">Saving Models Using Pickle</h2>

In [ ]:
import pickle

# logistic_grid_model
pickle.dump(logistic_grid, open("logistic_grid_model", "wb"))

# svc_grid_model
pickle.dump(svc_grid_model, open("svc_grid_model", "wb"))

# rf_grid_model
pickle.dump(rf_grid_model, open("rf_grid_model", "wb"))

# ada_grid_model
pickle.dump(ada_grid_model, open("ada_grid_model", "wb"))

# gb_grid_model
pickle.dump(gb_grid, open("gb_grid_model", "wb"))

# xgb_grid_model
pickle.dump(xgb_grid_model, open("xgb_grid_model", "wb"))

# lgbm_grid_model
pickle.dump(lgbm_grid_model, open("lgbm_grid_model", "wb"))

# cat_grid_model
pickle.dump(cat_grid_model, open("cat_grid_model", "wb"))

<h1 id="compare-models" style="color: mediumseagreen;">Comparing Models</h1>

In [ ]:
# Loading grid models using pickle

import pickle

# logistic_grid_model
logistic_grid = pickle.load(open("logistic_grid_model", "rb"))

# svc_grid_model
svc_grid_model = pickle.load(open("svc_grid_model", "rb"))

# rf_grid_model
rf_grid_model = pickle.load(open("rf_grid_model", "rb"))

# ada_grid_model
ada_grid_model = pickle.load(open("ada_grid_model", "rb"))

# gb_grid_model
gb_grid = pickle.load(open("gb_grid_model", "rb"))

# xgb_grid_model
xgb_grid_model = pickle.load(open("xgb_grid_model", "rb"))

# lgbm_grid_model
lgbm_grid_model = pickle.load(open("lgbm_grid_model", "rb"))

# cat_grid_model
cat_grid_model = pickle.load(open("cat_grid_model", "rb"))

In [ ]:
compare = pd.DataFrame({"Model": ["Logistic", "SVC", "RandomForest", "AdaBoost", "GradientBoost", "XGBoost", "LightGBM", "CatBoost"],
                        "Accuracy": [logistic_accuracy_test, svc_accuracy_test, rf_accuracy_test, ada_accuracy_test, gb_accuracy_test, xgb_accuracy_test, lgbm_accuracy_test, cat_accuracy_test],
                        "F1": [logistic_f1_test, svc_f1_test, rf_f1_test, ada_f1_test, gb_f1_test, xgb_f1_test, lgbm_f1_test, cat_f1_test],
                        "Recall": [logistic_recall_test, svc_recall_test, rf_recall_test, ada_recall_test, gb_recall_test, xgb_recall_test, lgbm_recall_test, cat_recall_test]
                       }
                      )

def labels(ax):
    for p in ax.patches:
        width = p.get_width()                        # get bar length
        ax.text(width,                               # set the text at 1 unit right of the bar
                p.get_y() + p.get_height() / 2,      # get Y coordinate + X coordinate / 2
                '{:1.3f}'.format(width),             # set variable to display, 2 decimals
                ha = 'left',                         # horizontal alignment
                va = 'center')                       # vertical alignment
    
plt.figure(figsize=(16,16))

plt.subplot(411)
compare = compare.sort_values(by="F1", ascending=False)
ax=sns.barplot(x="F1", y="Model", data=compare, palette="Blues_d")
labels(ax)

plt.subplot(412)
compare = compare.sort_values(by="Accuracy", ascending=False)
ax=sns.barplot(x="Accuracy", y="Model", data=compare, palette="Blues_d")
labels(ax)

plt.subplot(413)
compare = compare.sort_values(by="Recall", ascending=False)
ax=sns.barplot(x="Recall", y="Model", data=compare, palette="Blues_d")
labels(ax)

plt.show()

According to these results, it was decided to create the final model with the **SVC model**.

<h1 id="final-model" style="color: mediumseagreen;">Final Model & Prediction</h1>

<h2 id="fm1" style="color: limegreen;">Final Model</h2>

**param_grid created for SVC GridSearch:**

```python
param_grid = {
    'svc__C': [0.1, 0.5, 1],
    'svc__kernel': ['linear', 'rbf', 'sigmoid'],
    'svc__gamma': ["scale", "auto", 0.1, 1],
    'svc__degree': [1, 2, 3]
}
```

Comparing the results of the SVC grid model with the results of models created by changing a few parameter values.

If a better result is obtained by changing a few parameter values, that model will be the final model.

In [ ]:
# scores for svc_grid_model
eval_metric(svc_grid_model, X_train, y_train, X_test, y_test)

In [ ]:
operations = [ ("transformer", column_transformer), 
              ("svc", SVC(C=0.1, degree=1, gamma='scale', kernel='linear', random_state=42, probability=True))]

final_model = Pipeline(steps=operations)

final_model.fit(X_train,y_train)

In [ ]:
eval_metric(final_model, X_train, y_train, X_test, y_test)
# f1 score of hispanic (class 2) stayed the same as grid_model

In [ ]:
operations = [ ("transformer", column_transformer), 
              ("svc", SVC(C=0.2, degree=1, gamma='scale', kernel='linear', random_state=42, probability=True))]

final_model = Pipeline(steps=operations)

final_model.fit(X_train,y_train)

In [ ]:
eval_metric(final_model, X_train, y_train, X_test, y_test)
# f1 score of hispanic (class 2) increased compared to the grid model score.

In [ ]:
operations = [ ("transformer", column_transformer), 
              ("svc", SVC(C=0.3, degree=1, gamma='scale', kernel='linear', random_state=42, probability=True))]

final_model = Pipeline(steps=operations)

final_model.fit(X_train,y_train)

In [ ]:
eval_metric(final_model, X_train, y_train, X_test, y_test)
# f1 score of hispanic (class 2) increased compared to other models' scores.

In [ ]:
# saving final_model

import pickle
pickle.dump(final_model, open("final_model", "wb"))

<h2 id="fm2" style="color: limegreen;">Prediction</h2>

In [ ]:
# loading final model for prediction

import pickle
new_model = pickle.load(open("final_model", "rb"))

In [ ]:
my_dict = {
    'subjectid': [29600, 29601],
    'abdominalextensiondepthsitting': [240, 245],
    'acromialheight': [1450, 1387],
    'acromionradialelength': [330, 321],
    'anklecircumference': [220, 216],
    'axillaheight': [1370, 1070],
    'balloffootcircumference': [220, 275],
    'balloffootlength': [120, 104],
    'biacromialbreadth': [360, 391],
    'bicepscircumferenceflexed': [320, 307],
    'bicristalbreadth': [290, 245],
    'bideltoidbreadth': [420, 400],
    'bimalleolarbreadth': [90, 78],
    'bitragionchinarc': [380, 297],
    'bitragionsubmandibulararc': [400, 412],
    'bizygomaticbreadth': [130, 128],
    'buttockcircumference': [960, 968],
    'buttockdepth': [340, 339],
    'buttockheight': [940, 938],
    'buttockkneelength': [620, 623],
    'buttockpopliteallength': [490, 476],
    'calfcircumference': [400, 385],
    'cervicaleheight': [1450, 1468],
    'chestbreadth': [340, 367],
    'chestcircumference': [1020, 1038],
    'chestdepth': [340, 360],
    'chestheight': [1350, 1259],
    'crotchheight': [890, 870],
    'crotchlengthomphalion': [420, 415],
    'crotchlengthposterioromphalion': [450, 470],
    'earbreadth': [50, 53],
    'earlength': [60, 64],
    'earprotrusion': [25, 27],
    'elbowrestheight': [240, 248],
    'eyeheightsitting': [820, 821],
    'footbreadthhorizontal': [90, 93],
    'footlength': [270, 315],
    'forearmcenterofgriplength': [260, 302],
    'forearmcircumferenceflexed': [280, 301],
    'forearmforearmbreadth': [80, 86],
    'forearmhandlength': [450, 462],
    'functionalleglength': [1020, 1035],
    'handbreadth': [90, 96],
    'handcircumference': [210, 232],
    'handlength': [190, 192],
    'headbreadth': [150, 165],
    'headcircumference': [570, 603],
    'headlength': [190, 201],
    'heelanklecircumference': [230, 235],
    'heelbreadth': [90, 102],
    'hipbreadth': [390, 395],
    'hipbreadthsitting': [410, 400],
    'iliocristaleheight': [920, 936],
    'interpupillarybreadth': [60, 75],
    'interscyei': [360, 365],
    'interscyeii': [380, 385],
    'kneeheightmidpatella': [500, 510],
    'kneeheightsitting': [470, 478],
    'lateralfemoralepicondyleheight': [460, 562],
    'lateralmalleolusheight': [90, 98],
    'lowerthighcircumference': [500, 514],
    'mentonsellionlength': [120, 115],
    'neckcircumference': [390, 402],
    'neckcircumferencebase': [410, 430],
    'overheadfingertipreachsitting': [2400, 2540],
    'palmlength': [90, 98],
    'poplitealheight': [440, 432],
    'radialestylionlength': [260, 265],
    'shouldercircumference': [1100, 1152],
    'shoulderelbowlength': [340, 325],
    'shoulderlength': [520, 452],
    'sittingheight': [890, 745],
    'sleevelengthspinewrist': [750, 785],
    'sleeveoutseam': [650, 625],
    'span': [1900,1924],
    'stature': [1760, 1920],
    'suprasternaleheight': [1380, 1250],
    'tenthribheight': [1320, 1300],
    'thighcircumference': [580, 586],
    'thighclearance': [400,404],
    'thumbtipreach': [920, 900],
    'tibialheight': [440, 415],
    'tragiontopofhead': [160,175],
    'trochanterionheight': [910,905],
    'verticaltrunkcircumferenceusa': [1250,1120],
    'waistbacklength': [380,365],
    'waistbreadth': [320, 317],
    'waistcircumference': [860,875],
    'waistdepth': [290,300],
    'waistfrontlengthsitting': [400,415],
    'waistheightomphalion': [870,865],
    'weightkg': [750, 830],
    'wristcircumference': [170,168],
    'wristheight': [90,86],
    'Gender': ['Male', 'Male'],
    'Date': ['10-Oct-10', '18-Jun-10'],
    'Installation': ['Fort Hood', 'Fort Lee'],
    'Component': ['Regular Army', 'Army National Guard'],
    'Branch': ['Combat Arms', 'Combat Support'],
    'PrimaryMOS': ['19D', '68W'],
    'SubjectNumericRace': [1, 3],
    'SubjectsBirthLocation': ['Texas', 'Alaska'],
    'Ethnicity': ['Mexican', 'Colombian'],
    'Age': [28, 35],
    'Heightin': [69, 76],
    'Weightlbs': [165, 183],
    'WritingPreference': ['Right hand', 'Left hand']
}

In [ ]:
new_data = pd.DataFrame(my_dict)
new_data

**The process of carrying out the data preparation processes that are carried out before the model is created, before the prediction process.**

In [ ]:
new_data.drop(columns=["subjectid", "SubjectNumericRace", "Ethnicity", "Heightin", "Weightlbs", "Date", "Installation", "Component", "Branch", "PrimaryMOS"],
        inplace=True
       )

In [ ]:
def group_by_region(race):
    
    usa_states = [
        "California", "New York", "Texas", "Indiana", "Oklahoma", "Illinois", "Florida", "Minnesota", "Michigan", 
        "Mississippi", "Georgia", "Alabama", "South Carolina", "North Carolina", "Ohio", "Louisiana", "Pennsylvania", 
        "Virginia", "Massachusetts", "Wisconsin", "New Jersey", "Arizona","Missouri","Colorado","Maryland","Tennessee","Kentucky","Washington","Kansas"
        ,"South Dakota", "Iowa", "Arkansas", "Connecticut", "Jamaica", "Nebraska", "Hawaii", "Utah", "West Virginia", "Nevada", "Idaho", "Rhode Island",
        "District of Columbia", "North Dakota","Maine", "Delaware", "New Mexico", "New Hampshire", "Vermont", "Montana", "Alaska", "Wyoming", 
        "US Virgin Islands", "United States"
    ]
    latin_america = [
        "Puerto Rico", "Mexico", "Jamaica", "Dominican Republic", "Colombia", "Haiti", "Panama", "Guam", 
        "Guyana", "Brazil", "Peru", "El Salvador", "Ecuador", "Barbados", "Grenada", "Honduras", "Nicaragua", "Cuba", 
        "Romania", "Liberia", "Portugal", "Bulgaria", "Chile", "Argentina", "Belize", "Costa Rica", "Palau", "Cameroon", 
        "Paraguay", "Venezuela", "Fiji", "Trinidad and Tobago", "Antigua and Barbuda", "Ghana", "Togo", "Ivory Coast", 
        "Guatemala", "Senegal"
    ]
    europe = [
        "Germany", "United Kingdom", "Russia", "France", "Italy", "Poland", "Belgium", "Netherlands", "Romania", 
        "Israel", "Spain", "Denmark"
    ]
    asia = [
        "South Korea", "China", "Japan", "Vietnam", "India", "Nepal", "Taiwan", "Iran", "Sri Lanka", "Korea"
    ]
    africa = [
        "South Africa", "Nigeria", "Liberia", "Kenya", "Ethiopia"
    ]
    

    if race in usa_states:
        return "USA and States"
    elif race in latin_america:
        return "Latin America and Caribbean"
    elif race in europe:
        return "Europe"
    elif race in asia:
        return "Asia"
    elif race in africa:
        return "Africa"
    else:
        return "Other"


new_data['SubjectsBirthRegion'] = new_data['SubjectsBirthLocation'].apply(group_by_region)
new_data.drop("SubjectsBirthLocation", axis = 1, inplace = True)

In [ ]:
# Converting stature and weight values ​​into cm and kg
new_data['stature'] = new_data['stature'] / 10
new_data['weightkg'] = new_data['weightkg'] / 10

In [ ]:
prediction = new_model.predict(new_data)
prediction

In [ ]:
new_data["prediction"] = prediction
new_data

<h1 id="smote" style="color: mediumseagreen;">SMOTE</h1>

[About SMOTE](https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/)

**SMOTE (Synthetic Minority Over-sampling Technique)**

**Purpose**

SMOTE is an oversampling technique used to balance imbalanced datasets. Imbalanced datasets occur when one class significantly outnumbers the others. In such cases, the minority class (the class with fewer examples) is often overlooked, and the model performs better at predicting the majority class. SMOTE addresses this imbalance by generating new "synthetic" samples for the minority class.

**Applications**

- **Credit Risk Assessment**: Fraudulent cases are often rare, with most transactions being legitimate.
- **Medical Diagnosis**: Diseases that are rare result in an imbalance between healthy and diseased examples.
- **Customer Churn Analysis**: Most customers continue using a service, while only a few decide to leave.
- **Anomaly Detection**: Normal cases are usually more frequent, whereas anomalies (e.g., security breaches) are rare.

**Importance for Imbalanced Data**

Imbalanced datasets can negatively impact the performance of classification models. While the model may perform well at predicting the majority class, it often fails to accurately predict the minority class. This can be problematic, especially if the minority class is more critical (e.g., fraud detection, rare disease diagnosis). SMOTE is used to mitigate these challenges.

**How It Works**

1. A random sample from the minority class is selected.
2. The k nearest neighbors of this sample are identified.
3. One of these k neighbors is randomly selected.
4. A random interpolation factor is generated.
5. A new "synthetic" sample is created using this factor.

**Usage**

In Python, SMOTE is commonly implemented using the `imbalanced-learn` library.

**What is the Interpolation Factor?**
The interpolation factor in SMOTE is a weighting factor used to create new "synthetic" samples. The idea is to generate a new sample between a randomly selected minority class sample and one of its k nearest neighbors. This factor determines how "close" the synthetic sample is to the original or neighboring sample. It enhances the flexibility of the SMOTE algorithm, spreading minority class samples across a wider range and improving the model's generalization.

In [ ]:
print(y_train.value_counts())

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_encoded = column_transformer.fit_transform(X_train)
X_test_encoded = column_transformer.transform(X_test)

In [ ]:
from imblearn.over_sampling import SMOTE

sampling_strategy = {0: 3034, 1: 2000, 2: 2000}

over = SMOTE(sampling_strategy=sampling_strategy)
X_train_resampled, y_train_resampled = over.fit_resample(X_train_encoded, y_train_encoded)

steps = [('scaler', StandardScaler()), ("SVC", SVC(C=0.3, degree=1, gamma='scale', kernel='linear', random_state=42, probability=True))]

log_pipe_smote = Pipeline(steps=steps)

log_pipe_smote.fit(X_train_resampled, y_train_resampled)

In [ ]:
y_train_resampled_series = pd.Series(y_train_resampled)

value_counts = y_train_resampled_series.value_counts()
print(value_counts)

In [ ]:
eval_metric(log_pipe_smote, X_train_resampled, y_train_resampled, X_test_encoded, y_test_encoded)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

operations = [("scaler",StandardScaler()), ("SVC", SVC(random_state=42, probability=True))]

log_pipe_smote = Pipeline(steps=operations)

param_grid = {
    'SVC__C': [0.1, 0.5, 1],
    'SVC__kernel': ['linear', 'rbf', 'sigmoid'],
    'SVC__gamma': ["scale", "auto", 0.1, 1],
    'SVC__degree': [1, 2, 3]
}

smote_gridmodel = RandomizedSearchCV(log_pipe_smote, param_grid, scoring='recall_weighted', cv=10, n_jobs=-1, verbose=0)
smote_gridmodel.fit(X_train_resampled, y_train_resampled)

In [ ]:
# saving smote_grid_model

import pickle
pickle.dump(smote_gridmodel, open("smote_grid_model", "wb"))

In [ ]:
# loading smote_grid_model

import pickle
smote_gridmodel = pickle.load(open("smote_grid_model", "rb"))

In [ ]:
eval_metric(smote_gridmodel, X_train_resampled, y_train_resampled, X_test_encoded, y_test_encoded)

<h1 id="conclusion" style="color: mediumseagreen;">Conclusion</h1>

This study demonstrates the feasibility of predicting the ethnic backgrounds of U.S. military personnel based on their body measurements. By focusing on white, black, and Hispanic groups, the modeling processes have provided a solid foundation for evaluating the accuracy of classification algorithms. At the end of the project, SMOTE (Synthetic Minority Over-sampling Technique) was applied to address data imbalances and improve classification performance. These findings can serve as a reference for future studies in demographic analysis and related fields.

<p style="background-color:mediumseagreen;font-family:newtimeroman;font-size:200%;color:white;text-align:center;border-radius:60px 20px;"><b>THANK YOU!</b></p>